# Cynbit Pre-Sales Copilot: Colab backup demo
Use this notebook if the Streamlit link is unavailable. It runs the same pipeline as the app: **extract → retrieve (RAG) → score & route → grounded draft → guardrail**.

Runtime: CPU is fine. Takes about 2 minutes end to end.

In [ ]:
!git clone -q https://github.com/kight7/Cynbit-POC.git
%cd Cynbit-POC
!pip install -q -r requirements.txt

### API key (optional)
Add `GROQ_API_KEY` (free at console.groq.com) or `GEMINI_API_KEY` under **Colab → 🔑 Secrets**, or paste one when prompted. Leave it blank to run the offline mode.

In [ ]:
import os, getpass
try:
    from google.colab import userdata
    for k in ("GROQ_API_KEY", "GEMINI_API_KEY"):
        try:
            os.environ[k] = userdata.get(k)
        except Exception:
            pass
except ImportError:
    pass
if not (os.environ.get("GROQ_API_KEY") or os.environ.get("GEMINI_API_KEY")):
    k = getpass.getpass("GROQ_API_KEY (Enter to skip = offline mode): ")
    if k:
        os.environ["GROQ_API_KEY"] = k

In [ ]:
from copilot.pipeline import Copilot
cp = Copilot()
inquiry = """Hi, I'm Ritika, co-founder of CareQ, a telehealth startup in Bangalore (12 people).
Patients describe symptoms in our app and our two on-call doctors can't tell which ones are serious.
We want an AI layer that ranks incoming cases by urgency. We have about $35k and want a pilot within 6 weeks."""
r = cp.run(inquiry)
print("Mode:", r.mode, "| latency:", r.latency_ms)
print("\nLEAD SCORE:", r.score.total, r.score.tier.upper(), "→", r.route.team)
for b in r.score.breakdown:
    print(f"  {b.points:>2}/{b.max_points}  {b.label}: {b.reason}")
print("\nTOP CASE STUDIES:", [(c.case_id, c.client, round(c.confidence, 2)) for c in r.retrieved])

In [ ]:
from IPython.display import Markdown, display
display(Markdown(r.proposal_markdown))
print("Guardrail:", r.guardrail.model_dump())

### Extracted fields (JSON)

In [ ]:
print(r.fields.model_dump_json(indent=2))

### No-RAG baseline (needs an API key): shows why grounding matters

In [ ]:
if r.mode != "offline":
    b = cp.run(inquiry, use_rag=False)
    display(Markdown(b.proposal_markdown))
    print("Unsupported numbers in baseline:", b.guardrail.unverified_numbers)
else:
    print("Skipped: no API key")

### Full evaluation on the 30 labelled inquiries

In [ ]:
!python -m eval.run_eval --sleep 1
display(Markdown(open("eval/results/summary_llm.md" if os.environ.get("GROQ_API_KEY") or os.environ.get("GEMINI_API_KEY") else "eval/results/summary_offline.md").read()))